# Module 1 - Topic 6: Testing for Hallucination & Verification Habits
**AI Application Development - Developer Track (NIGCOMSAT)**

Two small, independent checks -- not a scripted "watch it fail" demo. This is what we actually found when we tested it, and why the test still matters either way. One concept per cell; run them in order.

## Import & Setup
Same setup as Topic 5 -- this notebook is self-contained.

In [9]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
)
MODEL_NAME = os.getenv("MODEL_NAME")

print("Client ready. Model:", MODEL_NAME)

Client ready. Model: google/gemma-4-E4B-it


## A Short Shift Note
Just enough text to contain the trap: two timestamps, and a rounded summary claim. No severity, no customers affected, no root cause -- this topic asks one question, so the document carries no more than that needs.

In [8]:
SHIFT_NOTE = """*** SYNTHETIC DOCUMENT - FOR TRAINING USE ONLY - NOT A REAL NIGCOMSAT RECORD ***

NOC Shift Note - Incident NOC-INC-2026-0417

02:14 - Loss-of-carrier alarm raised on link KU-HUB1-GW3.
03:52 - Full service restored after the faulty amplifier (HPA-2) was bypassed.

Summary: The outage lasted approximately two hours.
"""

print(SHIFT_NOTE)

*** SYNTHETIC DOCUMENT - FOR TRAINING USE ONLY - NOT A REAL NIGCOMSAT RECORD ***

NOC Shift Note - Incident NOC-INC-2026-0417

02:14 - Loss-of-carrier alarm raised on link KU-HUB1-GW3.
03:52 - Full service restored after the faulty amplifier (HPA-2) was bypassed.

Summary: The outage lasted approximately two hours.



## Two Questions, Two Kinds of Hallucination
No schema for either. First: a question with a precise correct answer the model might replace with a vaguer one. Second: a question with no answer in the document at all, which the model might invent anyway.

In [10]:
duration_response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {
            "role": "user",
            "content": (
                "Based on the shift note below, how long did the outage last, "
                "in minutes? Reply with just the number.\n\n" + SHIFT_NOTE
            ),
        },
    ],
)
duration_answer = duration_response.choices[0].message.content
print("Duration question -- raw answer:", repr(duration_answer))

Duration question -- raw answer: '98'


In [20]:
manufacturer_response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {
            "role": "user",
            "content": (
                "Based on the shift note below, what company manufactured the "
                "failed amplifier? Reply with just the company name.\n\n" + SHIFT_NOTE
            ),
        },
    ],
)
manufacturer_answer = manufacturer_response.choices[0].message.content
print("Manufacturer question -- raw answer:", repr(manufacturer_answer))
print()
print("The shift note never names a manufacturer. A good answer says so;")
print("an invented company name here would be a fabrication.")

Manufacturer question -- raw answer: 'Not mentioned'

The shift note never names a manufacturer. A good answer says so;
an invented company name here would be a fabrication.


## Building the Answer Key by Hand
We don't ask the model what the right answer is -- we compute it ourselves: 03:52 minus 02:14.

In [27]:
import re

KNOWN_DURATION_MINUTES = 98  # 03:52 - 02:14, computed by hand

def extract_minutes(text: str) -> int:
    """Pull the first integer out of the model's plain-text reply."""
    match = re.search(r"\d+", text)
    if not match:
        raise ValueError(f"No number found in model reply: {text!r}")
    return int(match.group())

reported_minutes = extract_minutes(duration_answer)
print("Model said:", reported_minutes, "minutes")
print("Known answer:", KNOWN_DURATION_MINUTES, "minutes")
print("Match:", reported_minutes == KNOWN_DURATION_MINUTES)

Model said: 98 minutes
Known answer: 98 minutes
Match: True


## We Actually Tested This
When this was tested during development -- six runs of the duration question, four runs of the manufacturer question -- the model answered **98 every time** and **"not mentioned" every time.** It never took either bait.

Run the two cells above a few times yourself (rerun them, don't just trust one pass) and see what you get. Neither result is guaranteed, and that's exactly the point of the next cell.

## Why You Still Write the Test
This model, on this gateway, right now, gets both questions right. None of these are guaranteed to stay true:

- The model behind `MODEL_NAME` can change without warning -- it already did once while this course was being built
- A cheaper or faster model swapped in later may not be as careful
- A small prompt change tomorrow could reintroduce the exact failure this test checks for

The test in the next section isn't there because we watched it fail. It's there because the failure is possible, and we want to know the moment it stops being avoided -- not after someone acts on a wrong number.

## Proving the Validator, Completely Separately
This second check needs no API call at all, no shift note, no relation to the questions above. We reuse the Topic 5 `IncidentExtract` class and prove its business-rule validator still works.

In [28]:
from pydantic import BaseModel, ValidationError, field_validator
from typing import Literal, Optional
import pytest


class IncidentExtract(BaseModel):
    incident_id: str
    severity: Literal["P1", "P2", "P3", "P4"]
    affected_link: str
    customers_affected: int
    root_cause: Optional[str] = None

    @field_validator("customers_affected")
    @classmethod
    def must_be_non_negative(cls, v):
        if v < 0:
            raise ValueError("customers_affected cannot be negative")
        return v


with pytest.raises(ValidationError):
    IncidentExtract(
        incident_id="NOC-INC-2026-0417",
        severity="P2",
        affected_link="KU-HUB1-GW3",
        customers_affected=-3,
        root_cause="unknown",
    )

print("Validator correctly rejected a negative customers_affected value.")

Validator correctly rejected a negative customers_affected value.


## Writing the Real Test File
Two tests: the duration check (needs the API) and the validator check (doesn't). The manufacturer question stays a live discussion exercise above, not an automated test -- there's no single correct refusal phrasing to assert against.

In [ ]:
%%writefile test_extraction.py
"""
Pytest suite for Module 1, Topic 6.

Run from the terminal in this folder:
    pytest test_extraction.py -v

test_duration_not_copied_from_summary needs a working .env (OPENAI_API_KEY,
MODEL_NAME, OPENAI_BASE_URL) -- it makes one plain, schema-free API call.
test_negative_customers_rejected does not call the model at all.
"""
import os
import re
from typing import Literal, Optional

import pytest
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, ValidationError, field_validator

load_dotenv()

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
)
MODEL_NAME = os.getenv("MODEL_NAME")

SHIFT_NOTE = """*** SYNTHETIC DOCUMENT - FOR TRAINING USE ONLY - NOT A REAL NIGCOMSAT RECORD ***

NOC Shift Note - Incident NOC-INC-2026-0417

02:14 - Loss-of-carrier alarm raised on link KU-HUB1-GW3.
03:52 - Full service restored after the faulty amplifier (HPA-2) was bypassed.

Summary: The outage lasted approximately two hours.
"""

KNOWN_DURATION_MINUTES = 98  # 03:52 minus 02:14, computed by hand


def extract_minutes(text: str) -> int:
    """Pull the first integer out of the model's plain-text reply."""
    match = re.search(r"\d+", text)
    if not match:
        raise ValueError(f"No number found in model reply: {text!r}")
    return int(match.group())


class IncidentExtract(BaseModel):
    incident_id: str
    severity: Literal["P1", "P2", "P3", "P4"]
    affected_link: str
    customers_affected: int
    root_cause: Optional[str] = None

    @field_validator("customers_affected")
    @classmethod
    def must_be_non_negative(cls, v):
        if v < 0:
            raise ValueError("customers_affected cannot be negative")
        return v


def test_duration_not_copied_from_summary():
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "user",
                "content": (
                    "Based on the shift note below, how long did the outage last, "
                    "in minutes? Reply with just the number.\n\n" + SHIFT_NOTE
                ),
            },
        ],
    )
    reported_minutes = extract_minutes(response.choices[0].message.content)
    assert reported_minutes == KNOWN_DURATION_MINUTES


def test_negative_customers_rejected():
    with pytest.raises(ValidationError):
        IncidentExtract(
            incident_id="NOC-INC-2026-0417",
            severity="P2",
            affected_link="KU-HUB1-GW3",
            customers_affected=-3,
            root_cause="unknown",
        )


## Running It From the Terminal

```
pytest test_extraction.py -v
```

Two tests, one command. If `test_duration_not_copied_from_summary` fails on any given run, that's not a broken demo -- that's the test doing its job.

## Recap & What's Next
- Hallucination comes from a model choosing the easy, wrong answer over the precise, correct one -- no schema needed to cause it or catch it
- The answer key comes from reading the source yourself, not from asking the model
- This model passed both traps we tried -- a fact worth knowing, not a reason to skip the test
- The validator test is offline, deterministic, and independent of everything else
- You test for failures you know are possible, not only ones you've personally seen happen

**Next:** Topic 7 moves from testing your own code to working safely in the shared Git repository -- branches, commits and pull requests, the workflow the Module 4 team build will require.